# Notebook 5: Cost, Latency, and Quality Analysis
## SmartCache ML: Multi-Policy Benchmark & Pareto Trade-offs

### 1. Problem Overview
An optimization is only genuinely effective if its benefits (cost reduction and latency reduction) are achieved without unacceptable degradation in response quality.

In this notebook, we execute a controlled benchmark over **identical test workloads** comparing five distinct policies:
1. **Policy 1: No-Cache Baseline (Always Large Model)**: Full capability, highest cost, highest latency.
2. **Policy 2: Exact-Match Cache + Always Large**: Tests exact string cache behavior.
3. **Policy 3: Semantic Cache + Always Large**: Tests semantic cache impact in isolation.
4. **Policy 4: Always Small Model (No Cache)**: Lowest cost without caching, but lower answer quality on complex tasks.
5. **Policy 5: SmartCache ML (Semantic Cache + Cost-Aware Router)**: Full integrated system.

---
### Learning Objectives
By the end of this notebook, you will be able to:
- Design a controlled benchmark replay workload.
- Record fine-grained per-request metrics: p50/p95 latency, cost, and token overlap quality.
- Compute percentage cost savings and latency reduction against the baseline.
- Plot cost vs quality and latency vs quality Pareto frontiers.
- Defend trade-offs with empirical data in an engineering interview.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from smartcache.config import settings
from smartcache.cache import SemanticCache, ExactMatchCache
from smartcache.router import ClassifierRouter
from smartcache.llm_provider import MockProvider
from smartcache.cost_tracker import CostTracker, RequestRecord
from smartcache.metrics import calculate_token_overlap_f1, compare_policies

print("Loaded benchmarking suite.")

### 2. Constructing the Benchmark Workload
We construct a representative replay stream containing canonical queries, paraphrases, and adversarial distractors.

In [ ]:
test_df = pd.read_csv(settings.data_processed_path / "test_queries.csv")
train_df = pd.read_csv(settings.data_processed_path / "train_queries.csv")

# Use a fixed combined workload of 50 queries with repeated paraphrases
workload = pd.concat([test_df, test_df[test_df['query_role'] == 'paraphrase']]).reset_index(drop=True)
print(f"Benchmark workload size: {len(workload)} requests.")

### 3. Executing the 5 Policies
Let's run the workload across all 5 configurations under identical conditions.

In [ ]:
def evaluate_policy(policy_name, use_semantic_cache=False, use_exact_cache=False, routing_policy="always_large"):
    tracker = CostTracker()
    provider = MockProvider()
    sem_cache = SemanticCache(db_path=settings.project_root / f"data/temp_{policy_name}.db", similarity_threshold=0.85)
    sem_cache.invalidate()
    exact_cache = ExactMatchCache()
    router = ClassifierRouter()
    
    for _, row in workload.iterrows():
        query = row['query_text']
        ref_ans = row['expected_answer']
        cacheable = row['is_cacheable']
        
        hit = False
        ans = ""
        sim_score = None
        lat = 0.0
        route_selected = "large"
        
        # 1. Cache phase
        if use_exact_cache and cacheable:
            look = exact_cache.lookup(query)
            if look.hit:
                hit = True
                ans = look.response
                lat = look.total_lookup_latency
                route_selected = "cache"
        elif use_semantic_cache and cacheable:
            look = sem_cache.lookup(query)
            sim_score = look.similarity_score
            if look.hit:
                hit = True
                ans = look.response
                lat = look.total_lookup_latency
                route_selected = "cache"
                
        # 2. Miss phase
        if not hit:
            if routing_policy == "always_large":
                route_selected = "large"
            elif routing_policy == "always_small":
                route_selected = "small"
            elif routing_policy == "router":
                dec = router.route(query)
                route_selected = dec.selected_route
                
            resp = provider.generate(query, model_type=route_selected)
            ans = resp.text
            lat = resp.latency_seconds
            
            # Store in cache
            if cacheable:
                if use_exact_cache:
                    exact_cache.store(query, ans)
                if use_semantic_cache:
                    sem_cache.store(query, ans)
                    
        cost = tracker.calculate_cost(route_selected, len(query)//4, len(ans)//4, is_cache_hit=hit)
        quality = calculate_token_overlap_f1(ans, ref_ans)
        
        rec = RequestRecord(
            request_id=f"{policy_name}_{row['query_id']}",
            query_text=query,
            cache_hit=hit,
            similarity_score=sim_score,
            selected_route=route_selected,
            is_mock=True,
            input_tokens=len(query)//4 if not hit else 0,
            output_tokens=len(ans)//4 if not hit else 0,
            latency_seconds=lat,
            estimated_cost_usd=cost,
            reference_answer=ref_ans,
            generated_answer=ans,
            quality_score=quality
        )
        tracker.record_request(rec)
        
    summary = tracker.get_summary()
    summary['mean_quality_f1'] = float(np.mean([r.quality_score for r in tracker.records]))
    return summary

In [ ]:
res_base = evaluate_policy("base_always_large", use_semantic_cache=False, routing_policy="always_large")
res_exact = evaluate_policy("exact_cache", use_exact_cache=True, routing_policy="always_large")
res_sem = evaluate_policy("semantic_cache", use_semantic_cache=True, routing_policy="always_large")
res_small = evaluate_policy("always_small", use_semantic_cache=False, routing_policy="always_small")
res_smart = evaluate_policy("smartcache_ml", use_semantic_cache=True, routing_policy="router")

results_table = pd.DataFrame([
    {"Policy": "1. Baseline (Always Large, No Cache)", **res_base},
    {"Policy": "2. Exact Cache + Large", **res_exact},
    {"Policy": "3. Semantic Cache + Large", **res_sem},
    {"Policy": "4. Always Small (No Cache)", **res_small},
    {"Policy": "5. SmartCache ML (Semantic + Router)", **res_smart},
])

results_table[['Policy', 'cache_hit_rate', 'total_cost_usd', 'mean_latency_ms', 'p95_latency_ms', 'mean_quality_f1']]

### 4. Visualizing Cost vs Quality Pareto Frontier

In [ ]:
plt.figure(figsize=(9, 5))
for _, row in results_table.iterrows():
    plt.scatter(row['total_cost_usd'], row['mean_quality_f1'], s=150, label=row['Policy'])

plt.xlabel("Total Inference Cost ($ USD)")
plt.ylabel("Answer Quality (Token F1 vs Reference)")
plt.title("Cost vs Quality Trade-off Across Policies")
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
plt.grid(True, linestyle='--', alpha=0.5)
plt.tight_layout()
plt.show()

### 5. Interview Concepts Checklist
1. **Pareto Optimality**: Demonstrating how SmartCache ML reaches the upper-left Pareto frontier (high quality, low cost) compared to naïve policies.
2. **P95 Tail Latency Reduction**: When semantic cache hits in < 15 ms, the entire latency distribution shifts left, cutting tail p95 latency dramatically.
3. **Quality Degradation Risk**: Why evaluating token overlap or LLM-as-a-judge quality is essential before deploying cost-saving routers in production.